# Adversary models: transfer, query budgets and perturbation shapes

**Module 06 — Adversarial Machine Learning**

The same classifier is attacked three ways here, and the differences matter more than the numbers. A
white-box attacker needs one gradient. A black-box attacker needs many queries — or a *surrogate* model
that transfers. And the shape of the budget ($L_\infty$ versus $L_0$) changes which attack is cheapest.

## Learning objectives

1. Measure attack transfer between two independently trained models.
2. Compare gradient-based, random and single-pixel attacks at comparable budgets.
3. Count queries as a cost, and relate it to detection opportunities (Module 18).
4. Choose the right norm for a domain and justify it.
5. State the threat model that each number does and does not support.

## Background

Three adversary models from Module 06:

* **White-box** — gradients available. Cost: one backward pass.
* **Black-box, query-based** — only outputs. Cost: hundreds to thousands of queries, each of which is
  visible to the defender.
* **Black-box, transfer** — attack a surrogate the attacker trained, then apply the perturbation to the
  target. Works because models learn similar features; fails when they do not.

The $L_0$ norm counts changed features. For tabular data (a transaction amount, an age) $L_0$ is often the
honest budget: an attacker changes three fields, not "everything by 2 %".

## Dependencies and installation

```bash
python -m pip install "torch>=2.4" --index-url https://download.pytorch.org/whl/cpu
python -m pip install "scikit-learn>=1.4" "numpy>=2.1" "pandas>=2.2" "matplotlib>=3.9"
```

CPU only, seconds of runtime.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

SEED = 4
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

digits = load_digits()
X = digits.images.astype("float32") / 16.0
y = digits.target.astype("int64")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
X_tr, X_te, y_tr, y_te = map(torch.from_numpy, (X_tr, X_te, y_tr, y_te))
loss_fn = nn.CrossEntropyLoss()
print("data ready:", tuple(X_tr.shape))

data ready: (1347, 8, 8)


**Expected output** — `data ready: (1347, 8, 8)`.

### Step 1 — Two models, so that transfer can be measured

Different depth and different seed: similar enough to learn the same features, different enough that
transfer is not trivial.

In [2]:
def make_model(hidden=64):
    return nn.Sequential(nn.Flatten(), nn.Linear(64, hidden), nn.ReLU(), nn.Linear(hidden, 10))


def train(model, epochs=12, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss_fn(model(X_tr), y_tr).backward()
        opt.step()
    model.eval()
    return model


def accuracy(model, x, y):
    with torch.no_grad():
        return float((model(x).argmax(1) == y).float().mean())


model_a = train(make_model(64))
torch.manual_seed(SEED + 1)
model_b = train(make_model(32))

print(f"model A clean accuracy {accuracy(model_a, X_te, y_te):.3f}")
print(f"model B clean accuracy {accuracy(model_b, X_te, y_te):.3f}")

model A clean accuracy 0.893
model B clean accuracy 0.860


**Expected output** — both models in the high 0.80s (about 0.89 and 0.86 here). Keep both numbers; an attack that drops A to 0.06 is only interesting if A started high.

### Step 2 — White-box, transfer, and random noise at the same budget

The random-noise row is the control condition. Without it, a reader cannot tell whether the attack worked
because of the gradient or because any change at that size hurts the model.

In [3]:
EPS = 0.2


def fgsm(model, x, y, eps=EPS):
    x = x.clone().requires_grad_(True)
    (grad,) = torch.autograd.grad(loss_fn(model(x), y), x)
    return (x + eps * grad.sign()).detach().clamp(0, 1)


x_white = fgsm(model_a, X_te, y_te)
x_random = (X_te + EPS * torch.from_numpy(rng.choice([-1.0, 1.0], size=X_te.shape).astype("float32"))).clamp(0, 1)

rows = [
    {"attack": "white-box on A", "target": "A", "accuracy": accuracy(model_a, x_white, y_te)},
    {"attack": "white-box on A (transfer)", "target": "B", "accuracy": accuracy(model_b, x_white, y_te)},
    {"attack": "random noise, same budget", "target": "A", "accuracy": accuracy(model_a, x_random, y_te)},
    {"attack": "clean", "target": "A", "accuracy": accuracy(model_a, X_te, y_te)},
]
table = pd.DataFrame(rows)
table["ASR"] = (1 - table.accuracy).round(3)
table.round(3)

,attack,target,accuracy,ASR
0,white-box on A,A,0.062,0.938
1,white-box on A (transfer),B,0.209,0.791
2,"random noise, same budget",A,0.824,0.176
3,clean,A,0.893,0.107


**Expected output** — transfer is weaker than the direct attack but far stronger than random noise. If the
transfer row were close to the clean row, the two models would have learned different features; if it were
close to the white-box row, an attacker would need no access at all. The truth is usually in between,
which is why transfer attacks are the practical threat.

### Step 3 — A single-pixel ($L_0$) attack

Change one pixel to its extreme value, searching over pixels and values, and keep the change that reduces
the true-class confidence most. This is what a physical attacker with a sticker or a tampered sensor can
actually do.

In [4]:
def one_pixel_attack(model, x, y, candidates=64):
    # Score each of the 64 pixels once with the pixel set to 0 and to 1.
    best = x.clone()
    best_conf = -1.0
    with torch.no_grad():
        base = torch.softmax(model(x), dim=1)[torch.arange(len(y)), y]
    for pix in range(candidates):
        for value in (0.0, 1.0):
            trial = x.clone()
            trial.view(len(x), -1)[:, pix] = value
            with torch.no_grad():
                conf = torch.softmax(model(trial), dim=1)[torch.arange(len(y)), y]
            improved = conf < base
            take = improved & (conf > best_conf)
            if take.any():
                best[take] = trial[take]
                best_conf = float(conf[take].max())
    return best


x_one = one_pixel_attack(model_a, X_te, y_te)
flipped = int((model_a(x_one).argmax(1) != y_te).sum())
print(f"single-pixel attack: {flipped} of {len(y_te)} images misclassified ({flipped/len(y_te):.3f})")

single-pixel attack: 49 of 450 images misclassified (0.109)


**Expected output** — a small but non-zero success rate (typically 0.02–0.15). One pixel out of 64 is a
severe constraint, yet it is enough to change a label, which is the argument for not trusting a model with
inputs an attacker can touch.

### Step 4 — Queries as a cost

A black-box attacker who cannot train a surrogate must query. Count how many queries a naive random search
needs to flip a label, and compare it with the single backward pass a white-box attacker spends.

In [5]:
def random_search_attack(model, x, y, max_queries=200, eps=EPS):
    queries = 0
    x_adv = x.clone()
    with torch.no_grad():
        original = model(x).argmax(1)
    for _ in range(max_queries):
        noise = torch.from_numpy(rng.choice([-1.0, 1.0], size=x.shape).astype("float32"))
        trial = (x + eps * noise).clamp(0, 1)
        queries += len(x)
        with torch.no_grad():
            pred = model(trial).argmax(1)
        improved = (pred != original) & (x_adv == x).all(dim=1).all(dim=1)
        x_adv[improved] = trial[improved]
    with torch.no_grad():
        final = model(x_adv).argmax(1)
    return x_adv, queries, float(((final != original) & (original == y)).float().mean())


x_rand, queries, asr = random_search_attack(model_a, X_te, y_te)
print(f"queries spent: {queries} for {asr:.3f} ASR at epsilon={EPS}")
print(f"white-box equivalent: 1 backward pass, ASR {1 - accuracy(model_a, x_white, y_te):.3f}")

queries spent: 90000 for 0.471 ASR at epsilon=0.2
white-box equivalent: 1 backward pass, ASR 0.938


**Expected output** — tens of thousands of queries for a much lower success rate than one gradient step.
That ratio is why query volume is a detection signal (Module 18) and why rate limits are a security
control rather than only a fairness mechanism (Module 16).

## Exercises

1. **Improve transfer.** Train the surrogate on 50 % of the data and attack the model trained on the other
   50 %. Does transfer improve or degrade, and why?
2. **Two-pixel attack.** Allow two pixels and report the success rate. Plot success against the number of
   pixels allowed.
3. **Choose a norm for tabular data.** For a fraud model with fields `amount`, `hour`, `merchant`, decide
   which norm is the honest attacker budget and defend the choice in three sentences.
4. **Bound the defender.** Suppose the API caps queries at 1 000 per identity per day. Recompute the
   random-search attack's expected success and state the assumption you had to make.

## Reflection questions

1. Which row of the comparison table would you show a product owner who says "our model is not public"?
2. Why is a single-pixel attack a better model of the physical world than an $L_\infty$ attack?
3. Query counting turns an attack into a detectable event. What does that imply about the value of
   logging requests (Module 18)?

## References

1. Papernot, N. et al., "The Limitations of Deep Learning in Adversarial Settings", EuroS&P 2016.
2. Su, J., Vargas, D. and Sakurai, K., "One Pixel Attack for Fooling Deep Neural Networks", 2017.
3. Papernot, N. et al., "Practical Black-Box Attacks against Machine Learning", AsiaCCS 2017.
4. Andriushchenko, M. et al., "Square Attack: a query-efficient black-box adversarial attack", 2020.